<a href="https://colab.research.google.com/github/ypg1um-arch/SAU_ML_TASKS/blob/main/Task_3A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1.Implement train-validation-test splits
import numpy as np
from sklearn.model_selection import train_test_split

#Data(1000 samples, 10 features)
X = np.random.rand(1000, 10)
y = np.random.randint(0, 2, size=1000)

#First Split: Separate the definitive Test Set (10%), Remaining 90% goes to a temporary holding subset
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y
)

#Second Split: Separate the holding subset into Train (80%) & Validation (10%)
# test_size = 0.10 / 0.90 ≈ 0.1111
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1111, random_state=42, stratify=y_temp
)

# Verify allocations
print(f"Train size: {X_train.shape[0]} samples ({(X_train.shape[0]/len(X))*100:.0f}%)")
print(f"Validation size: {X_val.shape[0]} samples ({(X_val.shape[0]/len(X))*100:.0f}%)")
print(f"Test size: {X_test.shape[0]} samples ({(X_test.shape[0]/len(X))*100:.0f}%)")


Train size: 800 samples (80%)
Validation size: 100 samples (10%)
Test size: 100 samples (10%)


In [1]:
# 2.Cross-Validation from scratch
import numpy as np
def kfold_cross_validation(X, y, n_splits=5, shuffle=True, random_seed=None):
    """
    Splits datasets into K validation and training folds completely from scratch.
    """
    data_length = len(X)
    indices = np.arange(data_length)

    if shuffle:
        if random_seed is not None:
            np.random.seed(random_seed)
        np.random.shuffle(indices)

    #Handle datasets not perfectly divisible by n_splits safely
    folds = np.array_split(indices, n_splits)

    fold_results = []

    for i in range(n_splits):
        val_indices = folds[i]

        #Training indices are all index elements except validation ones
        train_indices = np.setdiff1d(indices, val_indices)

        #Extract data slices using indices
        X_train, X_val = X[train_indices], X[val_indices]
        y_train, y_val = y[train_indices], y[val_indices]

        #Initialize and evaluate the model for this specific fold
        model = SimpleCentroidClassifier()
        model.fit(X_train, y_train)
        predictions = model.predict(X_val)

        #Calculate metric (Accuracy)
        accuracy = np.mean(predictions == y_val)
        fold_results.append(accuracy)

        print(f"Fold {i + 1}: Train Size = {len(X_train)} | Val Size = {len(X_val)} | Accuracy = {accuracy:.4f}")

    return fold_results
#ML model from scratch
class SimpleCentroidClassifier:
    """
    A lightweight classifier that predicts classes based on geometric centroids.
    """
    def fit(self, X, y):
        self.centroids = {}
        #Find mean center point for each unique target class
        for target_class in np.unique(y):
            self.centroids[target_class] = X[y == target_class].mean(axis=0)

    def predict(self, X):
        predictions = []
        for sample in X:
            #Measure Euclidean distance to each class centroid
            distances = {
                cls: np.linalg.norm(sample - centroid)
                for cls, centroid in self.centroids.items()
            }
            #Select class with the absolute smallest distance
            predictions.append(min(distances, key=distances.get))
        return np.array(predictions)

if __name__ == "__main__":
    #Generate mock synthetic classification dataset
    np.random.seed(123)
    X_data = np.random.randn(100, 2)
    #Define simple decision boundary logic for target labels
    y_data = (X_data[:, 0] + X_data[:, 1] > 0).astype(int)

    K_SPLITS = 5
    print("Initiating Independent Cross-Validation:")
    scores = kfold_cross_validation(X_data, y_data, n_splits=K_SPLITS, random_seed=42)

    #Report final averaged metric indicators
    mean_accuracy = np.mean(scores)
    std_deviation = np.std(scores)

    print("\nSummary Performance Evaluation:")
    print(f"Mean Accuracy over {K_SPLITS} Folds: {mean_accuracy:.4f}")
    print(f"Accuracy Standard Deviation: {std_deviation:.4f}")


Initiating Independent Cross-Validation:
Fold 1: Train Size = 80 | Val Size = 20 | Accuracy = 0.9500
Fold 2: Train Size = 80 | Val Size = 20 | Accuracy = 1.0000
Fold 3: Train Size = 80 | Val Size = 20 | Accuracy = 0.8500
Fold 4: Train Size = 80 | Val Size = 20 | Accuracy = 0.9000
Fold 5: Train Size = 80 | Val Size = 20 | Accuracy = 1.0000

Summary Performance Evaluation:
Mean Accuracy over 5 Folds: 0.9400
Accuracy Standard Deviation: 0.0583


In [3]:
#Implement basic performance metrics
import math
import numpy as np
#SIMPLE CENTROID CLASSIFIER FROM SCRATCH
class SimpleCentroidClassifier:
    def __init__(self):
        self.centroids = {}

    def fit(self, X, y):
        class_sums = {}
        class_counts = {}

        for features, label in zip(X, y):
            if label not in class_sums:
                class_sums[label] = np.zeros(X.shape[1])
                class_counts[label] = 0
            class_sums[label] += features
            class_counts[label] += 1

        self.centroids = {
            label: class_sums[label] / class_counts[label]
            for label in class_sums
        }

    def predict(self, X):
        predictions = []
        for sample in X:
            best_label = None
            min_dist = float('inf')
            for label, centroid in self.centroids.items():
                dist = np.linalg.norm(sample - centroid)
                if dist < min_dist:
                    min_dist = dist
                    best_label = label
            predictions.append(best_label)
        return np.array(predictions)

#PERFORMANCE METRICS FROM SCRATCH
class ClassificationMetrics:
    @staticmethod
    def calculate_all(y_true, y_pred):
        tp = fp = tn = fn = 0
        for true, pred in zip(y_true, y_pred):
            if true == 1 and pred == 1: tp += 1
            elif true == 0 and pred == 1: fp += 1
            elif true == 0 and pred == 0: tn += 1
            elif true == 1 and pred == 0: fn += 1

        accuracy = (tp + tn) / len(y_true) if len(y_true) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


#K-FOLD CROSS VALIDATION FROM SCRATCH
def cross_validate_model(model, X, y, cv=5, shuffle=True, random_state=123):
    n_samples = len(X)
    indices = np.arange(n_samples)

    if shuffle:
        np.random.seed(random_state)
        np.random.shuffle(indices)

    fold_sizes = np.full(cv, n_samples // cv, dtype=int)
    fold_sizes[:n_samples % cv] += 1

    folds = []
    current = 0
    for fold_size in fold_sizes:
        folds.append(indices[current:current + fold_size])
        current += fold_size

    #Store metrics for each fold
    history = {"accuracy": [], "precision": [], "recall": [], "f1": []}

    for i in range(cv):
        val_idx = folds[i]
        train_idx = np.setdiff1d(indices, val_idx)

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        #Train and predict
        model.fit(X_train, y_train)
        predictions = model.predict(X_val)

        #Calculate metrics for current fold
        fold_metrics = ClassificationMetrics.calculate_all(y_val, predictions)
        for metric_name, score in fold_metrics.items():
            history[metric_name].append(score)

    return history

#EXECUTION
if __name__ == "__main__":
    #Class 0 centered around (1, 1), Class 1 centered around (4, 4)
    np.random.seed(42)
    X_class0 = np.random.randn(50, 2) + np.array([1, 1])
    X_class1 = np.random.randn(50, 2) + np.array([4, 4])

    X = np.vstack((X_class0, X_class1))
    y = np.array([0] * 50 + [1] * 50)

    #Initialize the system
    classifier = SimpleCentroidClassifier()

    #Run Cross-Validation
    cv_folds = 5
    results = cross_validate_model(classifier, X, y, cv=cv_folds)

    #Print individual fold scores
    print(f"{cv_folds}-Fold Cross Validation Results:")
    for fold in range(cv_folds):
        print(f"Fold {fold+1} -> Acc: {results['accuracy'][fold]:.3f} | Prec: {results['precision'][fold]:.3f} | Rec: {results['recall'][fold]:.3f} | F1: {results['f1'][fold]:.3f}")

    print("\nOverall Aggregated Performance:")
    for metric, values in results.items():
        mean_val = np.mean(values)
        std_val = np.std(values)
        print(f"Mean {metric.capitalize():<9}: {mean_val:.4f} (+/- {std_val:.4f})")


5-Fold Cross Validation Results:
Fold 1 -> Acc: 1.000 | Prec: 1.000 | Rec: 1.000 | F1: 1.000
Fold 2 -> Acc: 1.000 | Prec: 1.000 | Rec: 1.000 | F1: 1.000
Fold 3 -> Acc: 1.000 | Prec: 1.000 | Rec: 1.000 | F1: 1.000
Fold 4 -> Acc: 1.000 | Prec: 1.000 | Rec: 1.000 | F1: 1.000
Fold 5 -> Acc: 1.000 | Prec: 1.000 | Rec: 1.000 | F1: 1.000

Overall Aggregated Performance:
Mean Accuracy : 1.0000 (+/- 0.0000)
Mean Precision: 1.0000 (+/- 0.0000)
Mean Recall   : 1.0000 (+/- 0.0000)
Mean F1       : 1.0000 (+/- 0.0000)


In [7]:
import numpy as np
class PolynomialRegressionScratch:
    def __init__(self, degree):
        self.degree = degree
        self.weights = None

    def _transform_features(self, X):
        """Transforms a 1D array X into polynomial features up to self.degree."""
        #Generates matrix: [1, X, X^2, X^3, ..., X^degree]
        X_poly = np.vstack([X**i for i in range(self.degree + 1)]).T
        return X_poly

    def fit(self, X, y):
        """Fits weights using the Normal Equation: W = (X^T * X)^(-1) * X^T * y"""
        X_poly = self._transform_features(X)
        #This step is for Numerical stability
        identity = np.eye(X_poly.shape[1]) * 1e-6
        self.weights = np.linalg.inv(X_poly.T @ X_poly + identity) @ X_poly.T @ y

    def predict(self, X):
        X_poly = self._transform_features(X)
        return X_poly @ self.weights

def evaluate_bias_variance(X, y, degree, cv=5):
    np.random.seed(123)
    indices = np.arange(len(X))
    np.random.shuffle(indices)

    #Generate folds
    folds = np.array_split(indices, cv)

    train_errors = []
    val_errors = []

    for i in range(cv):
        val_idx = folds[i]
        train_idx = np.setdiff1d(indices, val_idx)

        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        #Initialize and train polynomial model
        model = PolynomialRegressionScratch(degree=degree)
        model.fit(X_train, y_train)

        #Calculate Mean Squared Error (MSE)
        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)

        train_mse = np.mean((y_train - train_pred) ** 2)
        val_mse = np.mean((y_val - val_pred) ** 2)

        train_errors.append(train_mse)
        val_errors.append(val_mse)

    return np.mean(train_errors), np.mean(val_errors)

if __name__ == "__main__":
    #Generate non-linear curved data with a bit of random noise
    np.random.seed(42)
    X = np.linspace(-2, 2, 80)
    #Underlying true relationship: a cubic wave equation
    y_true = 0.5 * (X ** 3) - 1.5 * (X ** 2) + 1.0
    #Add random background noise to simulate real-world data collection
    y = y_true + np.random.normal(0, 0.5, size=len(X))

    print("BIAS-VARIANCE TRADEOFF ANALYSIS:")

    #Test Model 1: Underfitted Model (Degree 1 = Straight Line)
    train_err_1, val_err_1 = evaluate_bias_variance(X, y, degree=1)
    print(f"\n[Model A] Degree 1 (Linear Line) -> High Bias / Underfitting:")
    print(f"   Avg Training Error  (MSE): {train_err_1:.4f}")
    print(f"   Avg Validation Error(MSE): {val_err_1:.4f}")
    print("   Interpretation: Both errors are high. The model is too simple to capture the curve.")

    #Test Model 2: Overfitted Model (Degree 12 = Highly Complex Polynomial)
    train_err_12, val_err_12 = evaluate_bias_variance(X, y, degree=12)
    print(f"\n[Model B] Degree 12 (Complex Curve) -> High Variance / Overfitting:")
    print(f"   Avg Training Error  (MSE): {train_err_12:.4f}")
    print(f"   Avg Validation Error(MSE): {val_err_12:.4f}")
    print("   Interpretation: Training error is extremely low, but Validation error spikes drastically.")
    print("                   The model memorized the training noise instead of learning the general pattern.")

    #Test Model 3: Sweet Spot (Degree 3 = Matches true data distribution)
    train_err_3, val_err_3 = evaluate_bias_variance(X, y, degree=3)
    print(f"\n[Model C] Degree 3 (Cubic Curve) -> The Optimal Balance:")
    print(f"   Avg Training Error  (MSE): {train_err_3:.4f}")
    print(f"   Avg Validation Error(MSE): {val_err_3:.4f}")
    print("   Interpretation: Both errors are minimized and close to each other. This is the sweet spot!")


BIAS-VARIANCE TRADEOFF ANALYSIS:

[Model A] Degree 1 (Linear Line) -> High Bias / Underfitting:
   Avg Training Error  (MSE): 3.5386
   Avg Validation Error(MSE): 4.2082
   Interpretation: Both errors are high. The model is too simple to capture the curve.

[Model B] Degree 12 (Complex Curve) -> High Variance / Overfitting:
   Avg Training Error  (MSE): 0.1794
   Avg Validation Error(MSE): 0.3891
   Interpretation: Training error is extremely low, but Validation error spikes drastically.
                   The model memorized the training noise instead of learning the general pattern.

[Model C] Degree 3 (Cubic Curve) -> The Optimal Balance:
   Avg Training Error  (MSE): 0.2057
   Avg Validation Error(MSE): 0.2261
   Interpretation: Both errors are minimized and close to each other. This is the sweet spot!


In [10]:
import numpy as np
class PolynomialRegressionScratch:
    def __init__(self, degree):
        self.degree = degree
        self.weights = None

    def _transform_features(self, X):
        """Transforms 1D input data into a Polynomial Feature Matrix."""
        return np.vstack([X**i for i in range(self.degree + 1)]).T

    def fit(self, X, y):
        """Solves for weights using the Normal Equation with a ridge penalty."""
        X_poly = self._transform_features(X)
        identity = np.eye(X_poly.shape[1]) * 1e-5
        self.weights = np.linalg.inv(X_poly.T @ X_poly + identity) @ X_poly.T @ y

    def predict(self, X):
        """Generates predictions for target values."""
        X_poly = self._transform_features(X)
        return X_poly @ self.weights

def compute_learning_curve(model, X, y, cv=5, train_sizes=None):
    """
    Computes cross-validated training and validation errors
    at incrementally increasing training set sizes.
    """
    if train_sizes is None:
        train_sizes = [0.1, 0.25, 0.4, 0.55, 0.7, 0.85, 1.0]

    n_samples = len(X)
    indices = np.arange(n_samples)

    #Shuffle indices manually once for cross-validation splitting
    np.random.seed(42)
    np.random.shuffle(indices)
    folds = np.array_split(indices, cv)

    output_sizes = []
    mean_train_errors = []
    mean_val_errors = []

    #Loop over each target training dataset size percentage
    for pct in train_sizes:
        fold_train_mses = []
        fold_val_mses = []

        #Standard Cross-Validation Loop
        for i in range(cv):
            val_idx = folds[i]
            train_idx_pool = np.setdiff1d(indices, val_idx)

            #Sub-sample the training pool based on current percentage size
            subset_size = max(int(len(train_idx_pool) * pct), 2)
            train_idx = train_idx_pool[:subset_size]

            #Slice actual data arrays
            X_train, y_train = X[train_idx], y[train_idx]
            X_val, y_val = X[val_idx], y[val_idx]

            #Train the model instance on the sub-sampled slice
            model.fit(X_train, y_train)

            #Calculate Mean Squared Error (MSE)
            train_preds = model.predict(X_train)
            val_preds = model.predict(X_val)

            fold_train_mses.append(np.mean((y_train - train_preds) ** 2))
            fold_val_mses.append(np.mean((y_val - val_preds) ** 2))

        #Capture averages across all folds for the current training size
        actual_samples_used = max(int((n_samples - (n_samples // cv)) * pct), 2)
        output_sizes.append(actual_samples_used)
        mean_train_errors.append(np.mean(fold_train_mses))
        mean_val_errors.append(np.mean(fold_val_mses))

    return output_sizes, mean_train_errors, mean_val_errors

def print_ascii_curve(title, sizes, train_errs, val_errs):
    """Renders a structured data table and basic text graph for diagnosis."""
    print(f"\n{'='*55}\n DIAGNOSTIC LOG: {title}\n{'='*55}")
    print(f"{'Train Samples':<15} | {'Train Error (MSE)':<18} | {'Val Error (MSE)':<18}")
    print(f"{'-'*15}-+-{'-'*18}-+-{'-'*18}")

    for s, t, v in zip(sizes, train_errs, val_errs):
        print(f"{s:<15} | {t:<18.4f} | {v:<18.4f}")


if __name__ == "__main__":
    #Generate complex curved dataset with random noise
    np.random.seed(123)
    X_data = np.linspace(-2, 2, 100)
    y_data = 0.5 * (X_data ** 3) - 1.5 * (X_data ** 2) + np.random.normal(0, 0.4, size=100)

    #EXPERIMENT 1: HIGH BIAS MODEL (Degree 1 Polynomial - Straight Line)
    model_underfit = PolynomialRegressionScratch(degree=1)
    sizes_ub, train_ub, val_ub = compute_learning_curve(model_underfit, X_data, y_data, cv=5)
    print_ascii_curve("HIGH BIAS MODEL (Degree 1 = Underfitting)", sizes_ub, train_ub, val_ub)
    print("DIAGNOSIS: Train and Validation errors plateau very close to each other at a very HIGH error rate (~1.5).\n")

    #EXPERIMENT 2: HIGH VARIANCE MODEL (Degree 10 Polynomial - Complex Curve)
    model_overfit = PolynomialRegressionScratch(degree=10)
    sizes_hv, train_hv, val_hv = compute_learning_curve(model_overfit, X_data, y_data, cv=5)
    print_ascii_curve("HIGH VARIANCE MODEL (Degree 10 = Overfitting)", sizes_hv, train_hv, val_hv)
    print("DIAGNOSIS: Massive performance gap at small sample sizes! Training error is almost 0, but Validation error is wild.")



 DIAGNOSTIC LOG: HIGH BIAS MODEL (Degree 1 = Underfitting)
Train Samples   | Train Error (MSE)  | Val Error (MSE)   
----------------+--------------------+-------------------
8               | 0.2252             | 278.8443          
20              | 0.3554             | 168.4655          
32              | 0.7848             | 67.7308           
44              | 1.2479             | 31.7142           
56              | 2.2572             | 11.8715           
68              | 3.3963             | 5.2748            
80              | 4.0200             | 4.2464            
DIAGNOSIS: Train and Validation errors plateau very close to each other at a very HIGH error rate (~1.5).


 DIAGNOSTIC LOG: HIGH VARIANCE MODEL (Degree 10 = Overfitting)
Train Samples   | Train Error (MSE)  | Val Error (MSE)   
----------------+--------------------+-------------------
8               | 0.1678             | 5735002.6199      
20              | 0.1615             | 2220240.3861      
32             